# Phase 4 — Targeted Person Augmentation

Builds `final_dataset_v1` from `processed_v1`. Only Person-only training images are augmented, so no other class is augmented. The source datasets are preserved. No training is performed.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.augmentation import (
    build_final_dataset,
    reconstruct_augmentation_records,
    repair_generated_label_bounds,
    save_augmentation_charts,
    write_augmentation_report,
)
from weapon_threat_detection.config import load_config
from weapon_threat_detection.engineering import MERGED_CLASS_NAMES, calculate_class_weights, collect_dataset_statistics, write_yaml
from weapon_threat_detection.validation import build_integrity_report, summarize_records, validate_dataset, write_validation_csv

config = load_config(ROOT / 'configs' / 'project.yaml')
source = ROOT / config['final_augmentation']['source_dataset']
target = ROOT / config['final_augmentation']['target_dataset']
augmentation_config = ROOT / config['final_augmentation']['configuration']
reports_dir = ROOT / config['project']['reports_dir']
logger = configure_logger(ROOT / config['project']['logs_dir'], 'targeted_person_augmentation')
before = collect_dataset_statistics(source, MERGED_CLASS_NAMES, config['dataset']['splits'])
recovery = {'changed_files': 0, 'changed_annotations': 0}
if target.exists() and any(target.iterdir()):
    recovery = repair_generated_label_bounds(target)
    generated_records = reconstruct_augmentation_records(source, target)
    metadata = json.loads((target / 'metadata.json').read_text(encoding='utf-8'))
    result = {'metadata': metadata, 'generated_records': generated_records}
else:
    result = build_final_dataset(source, target, augmentation_config, config['dataset']['splits'])
after = collect_dataset_statistics(target, MERGED_CLASS_NAMES, config['dataset']['splits'])
weights = calculate_class_weights(after, MERGED_CLASS_NAMES)
charts = save_augmentation_charts(before, after, reports_dir)
final_records = validate_dataset(target, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
summary = summarize_records(final_records)
if any(status not in {'valid', 'valid_background'} for status in summary):
    raise RuntimeError(f'Final dataset validation failed: {summary}')
stamp = utc_timestamp()
augmentation_report = write_augmentation_report(result['generated_records'], reports_dir / f'augmentation_report_{stamp}.csv')
comparison_path = write_json(reports_dir / f'augmentation_before_after_{stamp}.json', {'before': before, 'after': after, 'person_annotation_delta': after['annotations_per_class']['Person'] - before['annotations_per_class']['Person'], 'person_image_delta': after['images_per_class']['Person'] - before['images_per_class']['Person'], 'generated_images': result['metadata']['generated_images'], 'generated_person_annotations': result['metadata']['generated_person_annotations'], 'bounds_recovery': recovery})
statistics_path = write_json(reports_dir / f'final_dataset_statistics_{stamp}.json', after)
weights_json = write_json(reports_dir / 'final_class_weights.json', weights)
weights_yaml = write_yaml(reports_dir / 'final_class_weights.yaml', weights)
validation_path = write_validation_csv(final_records, reports_dir / f'final_dataset_validation_{stamp}.csv')
integrity_path = write_json(reports_dir / f'final_dataset_integrity_{stamp}.json', build_integrity_report(final_records, config['dataset']['splits']))
summary_path = write_json(reports_dir / f'final_dataset_summary_{stamp}.json', {'source': str(source), 'target': str(target), 'source_modified': False, 'generated_class': 'Person', 'person_only_source_images': True, 'augmentation_config': str(augmentation_config), 'metadata': result['metadata'], 'before_after': str(comparison_path), 'statistics': str(statistics_path), 'weights_json': str(weights_json), 'weights_yaml': str(weights_yaml), 'validation': str(validation_path), 'integrity': str(integrity_path), 'charts': charts, 'bounds_recovery': recovery, 'validation_summary': summary, 'training_started': False})
logger.info('Generated %s Person-only augmented training images.', result['metadata']['generated_images'])
logger.info('Bounds recovery: %s', recovery)
logger.info('Final validation completed: %s', summary)
print({'final_dataset': str(target), 'generated_images': result['metadata']['generated_images'], 'generated_person_annotations': result['metadata']['generated_person_annotations'], 'bounds_recovery': recovery, 'validation': summary, 'completion_report': str(summary_path)})

/Users/mohamedtarek/weapon130/.venv/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


{'final_dataset': '/Users/mohamedtarek/weapon130/processed/final_dataset_v1', 'generated_images': 1222, 'generated_person_annotations': 3428, 'bounds_recovery': {'changed_files': 566, 'changed_annotations': 1660}, 'validation': {'valid': 119125, 'valid_background': 12342}, 'completion_report': '/Users/mohamedtarek/weapon130/reports/final_dataset_summary_20260723T095607Z.json'}
